# خادم LLM مجاني لمنارة على Kaggle GPU (30 ساعة/أسبوع)
1. شغّل هذه الخلايا بالترتيب على Kaggle (Accelerator: GPU T4).
2. انسخ رابط cloudflared العام.
3. ضعه في Vercel باسم `AGENT_LLM_URL` (مثال: `https://xxx.trycloudflare.com/v1`) + `AGENT_LLM_MODEL=Qwen/Qwen2.5-7B-Instruct-AWQ`.
ملاحظة: الرابط يتغير مع كل تشغيل — ثبّته في Vercel كل مرة (أو استخدم ngrok الثابت المجاني).

In [ ]:
!pip install -q vllm 2>&1 | tail -1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cf && chmod +x /tmp/cf
print('ready')

In [ ]:
import subprocess, threading, time
srv = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server','--model','Qwen/Qwen2.5-7B-Instruct-AWQ','--quantization','awq','--max-model-len','4096','--port','8000'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
cf = subprocess.Popen(['/tmp/cf','tunnel','--url','http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import re
url = None
start = time.time()
while time.time() - start < 120 and not url:
    line = cf.stdout.readline()
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line or '')
    if m: url = m.group(0)
print('PUBLIC URL:', url)
print('USE: AGENT_LLM_URL=' + (url + '/v1' if url else 'FAILED'))

In [ ]:
# اختبار سريع (من نفس النوتبوك)
import json, urllib.request
req = urllib.request.Request('http://localhost:8000/v1/chat/completions', data=json.dumps({'model':'Qwen/Qwen2.5-7B-Instruct-AWQ','messages':[{'role':'user','content':'قل: يعمل'}],'max_tokens':20}).encode(), headers={'Content-Type':'application/json'})
print(urllib.request.urlopen(req, timeout=300).read().decode()[:200])